`ensemble_comparisons` - accuracy фреймворка и его участников на MedMCQA...


In [1]:
from pathlib import Path
import os
_REPOSITORY_ROOT = next((p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / 'experiments/clinical_protocols').is_dir() and (p / 'data').is_dir()))
if os.name == 'nt' and (not str(_REPOSITORY_ROOT).startswith('\\\\?\\')):
    _REPOSITORY_ROOT = Path('\\\\?\\' + str(_REPOSITORY_ROOT))
_NOTEBOOK_DIR = _REPOSITORY_ROOT / 'experiments/external_benchmarks/medmcqa'
os.chdir(_NOTEBOOK_DIR)


In [2]:
import json
import os
from typing import Dict, List, Optional, Sequence
import pandas as pd
from scipy import stats
ROOT = os.path.abspath(os.getcwd())
DELPHI_DIR = 'panel_gpt120_organizer_qwen_deepseek'
SYSTEMS: List[tuple] = [('Delphi AC-DeepSeek (фреймворк)', os.path.join(DELPHI_DIR, 'delphi_medmcqa_noroles.csv'), 'delphi_consensus_selected_option'), ('qwen3-235b-a22b-fp8', os.path.join('baseline_qwen', 'baseline_medmcqa_qwen.csv'), 'baseline_selected_option'), ('deepseek-v4-flash', os.path.join('baseline_deepseekv4', 'baseline_medmcqa_deepseekv4.csv'), 'baseline_selected_option'), ('gpt-oss-120b (модератор)', os.path.join('baseline_gpt', 'baseline_medmcqa_gpt.csv'), 'baseline_selected_option')]
TRUTH_COLUMN = 'correct_option_letter'
VALID_OPTIONS = frozenset('ABCD')
FIRST_N = 100
PREPARED_CSV = os.path.join(ROOT, '..', 'initial_runs', 'medmcqa_prepared.csv')
SAMPLE_SIZE_FULL = 1000
RANDOM_SEED = 42
pd.set_option('display.width', 200)
print('ROOT:', ROOT)


ROOT: .\experiments\external_benchmarks\medmcqa


In [3]:
def read_run(path: str) -> pd.DataFrame:
    df = pd.read_csv(os.path.abspath(path), sep=';', encoding='utf-8-sig', low_memory=False)
    before = len(df)
    df = df[df['case_id'].notna()].drop_duplicates('case_id').reset_index(drop=True)
    dropped = before - len(df)
    if dropped:
        print(f'  {os.path.basename(os.path.dirname(path)):<62} отброшено артефактных строк: {dropped}')
    return df

def answer_series(df: pd.DataFrame, column: str) -> pd.Series:
    letters = df[column].astype(str).str.strip().str.upper().str[:1]
    return pd.Series(letters.values, index=df['case_id'].values)
print('Загрузка прогонов:')
predictions: Dict[str, pd.Series] = {}
truth: Optional[pd.Series] = None
case_order: Optional[List] = None
for label, path, column in SYSTEMS:
    run = read_run(os.path.join(ROOT, path))
    predictions[label] = answer_series(run, column)
    if truth is None:
        truth = answer_series(run, TRUTH_COLUMN)
        case_order = list(run['case_id'])
print()
print(f'Кейсов в прогоне: {len(case_order)}')
print(f'Систем загружено: {len(predictions)}')


Загрузка прогонов:


  panel_gpt120_organizer_qwen_deepseek                           отброшено артефактных строк: 6
  baseline_qwen                                                  отброшено артефактных строк: 1


  baseline_deepseekv4                                            отброшено артефактных строк: 1
  baseline_gpt                                                   отброшено артефактных строк: 1

Кейсов в прогоне: 1000
Систем загружено: 4


In [4]:
reference = set(case_order)
for label, series in predictions.items():
    same = set(series.index) == reference
    print(f'  {label:<32} кейсы совпадают с эталоном: {same}  (n={len(series)})')
print()
if os.path.exists(PREPARED_CSV):
    prepared = pd.read_csv(os.path.abspath(PREPARED_CSV), sep=';', encoding='utf-8-sig', low_memory=False).reset_index(drop=True)
    sample_full = prepared.sample(n=SAMPLE_SIZE_FULL, random_state=RANDOM_SEED).reset_index(drop=True)
    sample_head = prepared.sample(n=FIRST_N, random_state=RANDOM_SEED).reset_index(drop=True)
    print(f'Сверка с {os.path.relpath(PREPARED_CSV, ROOT)}:')
    print(f"  порядок строк прогона == порядок sample(1000, seed={RANDOM_SEED}): {case_order == list(sample_full['case_id'])}")
    print(f"  sample({FIRST_N}) == первые {FIRST_N} строк sample({SAMPLE_SIZE_FULL}): {list(sample_head['case_id']) == list(sample_full['case_id'][:FIRST_N])}")
else:
    print(f'Сверка пропущена: {PREPARED_CSV} недоступен.')
    print("Определение 'первых 100' опирается на порядок строк прогона.")


  Delphi AC-DeepSeek (фреймворк)   кейсы совпадают с эталоном: True  (n=1000)
  qwen3-235b-a22b-fp8              кейсы совпадают с эталоном: True  (n=1000)
  deepseek-v4-flash                кейсы совпадают с эталоном: True  (n=1000)
  gpt-oss-120b (модератор)         кейсы совпадают с эталоном: True  (n=1000)

Сверка пропущена: .\experiments\external_benchmarks\medmcqa\..\external_benchmarks/initial_runs\medmcqa_prepared.csv недоступен.
Определение 'первых 100' опирается на порядок строк прогона.


In [5]:
def accuracy_table(case_ids: Sequence, label_of_subset: str) -> pd.DataFrame:
    index = pd.Index(case_ids)
    reference_answers = truth.reindex(index)
    rows = []
    for label, series in predictions.items():
        answers = series.reindex(index)
        correct = answers == reference_answers
        invalid = ~answers.isin(VALID_OPTIONS)
        rows.append({'система': label, 'верно': int(correct.sum()), 'из': len(index), 'accuracy': float(correct.mean()), 'невалидных': int(invalid.sum())})
    table = pd.DataFrame(rows).sort_values('accuracy', ascending=False).reset_index(drop=True)
    table.insert(0, 'выборка', label_of_subset)
    return table

def show(table: pd.DataFrame, title: str) -> None:
    print('=' * 72)
    print(title)
    print('=' * 72)
    view = table.drop(columns=['выборка']).copy()
    view['верно'] = view['верно'].astype(str) + ' / ' + view['из'].astype(str)
    view = view.drop(columns=['из'])
    print(view.to_string(index=False, float_format=lambda v: f'{v:.4f}'))
    print()


In [6]:
first_cases = case_order[:FIRST_N]
all_cases = case_order
table_first = accuracy_table(first_cases, f'первые {FIRST_N}')
table_all = accuracy_table(all_cases, f'вся выборка {len(all_cases)}')
show(table_first, f'ПЕРВЫЕ {FIRST_N} КЕЙСОВ')
show(table_all, f'ВСЯ ВЫБОРКА, {len(all_cases)} КЕЙСОВ')


ПЕРВЫЕ 100 КЕЙСОВ
                       система    верно  accuracy  невалидных
Delphi AC-DeepSeek (фреймворк) 94 / 100    0.9400           0
           qwen3-235b-a22b-fp8 91 / 100    0.9100           0
             deepseek-v4-flash 89 / 100    0.8900           1
      gpt-oss-120b (модератор) 84 / 100    0.8400           0

ВСЯ ВЫБОРКА, 1000 КЕЙСОВ
                       система      верно  accuracy  невалидных
Delphi AC-DeepSeek (фреймворк) 881 / 1000    0.8810           0
             deepseek-v4-flash 879 / 1000    0.8790           5
           qwen3-235b-a22b-fp8 814 / 1000    0.8140          11
      gpt-oss-120b (модератор) 813 / 1000    0.8130           3



In [7]:
def mcnemar(system_a: str, system_b: str, case_ids: Sequence) -> Dict[str, float]:
    index = pd.Index(case_ids)
    reference_answers = truth.reindex(index)
    a_correct = predictions[system_a].reindex(index) == reference_answers
    b_correct = predictions[system_b].reindex(index) == reference_answers
    n10 = int((a_correct & ~b_correct).sum())
    n01 = int((~a_correct & b_correct).sum())
    p = stats.binomtest(n10, n10 + n01, 0.5).pvalue if n10 + n01 else 1.0
    return {'n': len(index), 'acc_a': float(a_correct.mean()), 'acc_b': float(b_correct.mean()), 'delta': float(a_correct.mean() - b_correct.mean()), 'n10': n10, 'n01': n01, 'p': float(p)}
FRAMEWORK = SYSTEMS[0][0]
baseline_labels = [label for label, _, _ in SYSTEMS[1:]]
rows = []
for subset_label, case_ids in ((f'первые {FIRST_N}', first_cases), (f'вся выборка {len(all_cases)}', all_cases)):
    index = pd.Index(case_ids)
    reference_answers = truth.reindex(index)
    best = max(baseline_labels, key=lambda lab: (predictions[lab].reindex(index) == reference_answers).mean())
    for label in baseline_labels:
        res = mcnemar(FRAMEWORK, label, case_ids)
        rows.append({'выборка': subset_label, 'против': label + (' (лучший baseline)' if label == best else ''), 'фреймворк': res['acc_a'], 'модель': res['acc_b'], 'delta': res['delta'], 'n10/n01': f"{res['n10']}/{res['n01']}", 'p': res['p']})
tests = pd.DataFrame(rows)
print(tests.to_string(index=False, float_format=lambda v: f'{v:.4f}'))


         выборка                                против  фреймворк  модель  delta n10/n01      p
      первые 100 qwen3-235b-a22b-fp8 (лучший baseline)     0.9400  0.9100 0.0300     5/2 0.4531
      первые 100                     deepseek-v4-flash     0.9400  0.8900 0.0500     6/1 0.1250
      первые 100              gpt-oss-120b (модератор)     0.9400  0.8400 0.1000    10/0 0.0020
вся выборка 1000                   qwen3-235b-a22b-fp8     0.8810  0.8140 0.0670   97/30 0.0000
вся выборка 1000   deepseek-v4-flash (лучший baseline)     0.8810  0.8790 0.0020   35/33 0.9036
вся выборка 1000              gpt-oss-120b (модератор)     0.8810  0.8130 0.0680   85/17 0.0000


In [8]:
summary = {'dataset': 'MedMCQA', 'n_full': len(all_cases), 'n_first': FIRST_N, 'systems': [label for label, _, _ in SYSTEMS], 'accuracy_first': json.loads(table_first.to_json(orient='records', force_ascii=False)), 'accuracy_all': json.loads(table_all.to_json(orient='records', force_ascii=False)), 'mcnemar_vs_baselines': json.loads(tests.to_json(orient='records', force_ascii=False))}
out_path = os.path.join(ROOT, 'analysis_external_accuracy.json')
with open(out_path, 'w', encoding='utf-8') as fh:
    json.dump(summary, fh, ensure_ascii=False, indent=2)
print('saved:', out_path)


saved: .\experiments\external_benchmarks\medmcqa\analysis_external_accuracy.json
